# 02 — Padding waste: fixed item-count vs length-sorted token-budget batching

The backend pads every batch to its longest member, so the compute cost of a
batch is `count * max_len` ("padded tokens"). **Waste** = padded tokens −
real tokens: work spent embedding padding.

This notebook prefers a **real corpus with real tokenizer lengths**
(`ag_news` train split via `datasets`, tokenized with MiniLM's tokenizer). If
`datasets`/`transformers` are not installed (they arrive with the `gpu`
extra), it falls back to the synthetic log-normal distribution from notebook
01 **and says so in the output** — the markdown numbers below are from that
synthetic fallback.

```
uv run --with jupyterlab --with matplotlib --with datasets jupyter lab
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from embedx.engine import make_batches

try:
    from datasets import load_dataset
    from transformers import AutoTokenizer

    rows = load_dataset("ag_news", split="train[:2000]")
    tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
    encoded = tokenizer([r["text"] for r in rows], truncation=False, padding=False)
    lengths = np.array([len(ids) for ids in encoded["input_ids"]])
    source = "ag_news train[:2000], real MiniLM token lengths"
except Exception as exc:  # noqa: BLE001 — any import/download failure -> fallback
    rng = np.random.default_rng(7)
    lengths = np.clip(rng.lognormal(mean=4.0, sigma=0.9, size=2000).astype(int), 10, 3000)
    source = (f"SYNTHETIC FALLBACK (log-normal, seed 7) — real corpus unavailable: "
              f"{type(exc).__name__}")

print(f"corpus: {source}")
real_tokens = int(lengths.sum())
print(f"n={len(lengths)}  real tokens={real_tokens}  median={np.median(lengths):.0f}  "
      f"p95={np.percentile(lengths, 95):.0f}  max={lengths.max()}")

In [ ]:
def padded_fixed(lengths, batch_size):
    # Fixed item-count batches in ARRIVAL order — what a naive server does.
    total = 0
    for start in range(0, len(lengths), batch_size):
        chunk = lengths[start:start + batch_size]
        total += len(chunk) * int(max(chunk))
    return total


def padded_budget(lengths, budget):
    # The real make_batches: stable ascending length sort + token budget.
    texts = ["x" * int(n) for n in lengths]
    return sum(len(batch) * max(len(t) for _, t in batch)
               for batch in make_batches(list(enumerate(texts)), budget))


def waste_ratio(padded):
    return (padded - real_tokens) / padded


print("fixed item-count batching (arrival order):")
for batch_size in (8, 32, 128):
    padded = padded_fixed(lengths, batch_size)
    print(f"  bs={batch_size:4d}: padded={padded:9d}  waste_ratio={waste_ratio(padded):.1%}")

budgets = [1024, 2048, 4096, 8192, 16384, 32768, 65536]
print("length-sorted token-budget batching:")
budget_waste = []
for budget in budgets:
    padded = padded_budget(lengths, budget)
    budget_waste.append(waste_ratio(padded))
    print(f"  budget={budget:6d}: padded={padded:9d}  waste_ratio={budget_waste[-1]:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.semilogx(budgets, [w * 100 for w in budget_waste], "o-",
            label="length-sorted, token budget")
for batch_size, style in [(8, ":"), (32, "--"), (128, "-.")]:
    ax.axhline(waste_ratio(padded_fixed(lengths, batch_size)) * 100,
               linestyle=style, color="grey", label=f"fixed bs={batch_size}")
ax.set_xlabel("max_batch_tokens (log scale)")
ax.set_ylabel("waste (% of padded tokens)")
ax.set_title("padding waste vs batch budget")
ax.legend()
plt.show()

## The numbers (synthetic fallback, seed 7)

| strategy | waste (fraction of compute spent on padding) |
|---|---|
| fixed bs=8 | **64.7%** |
| fixed bs=32 | **78.8%** |
| fixed bs=128 | **84.5%** |
| budget 1,024 | 1.0% |
| budget 4,096 | 5.0% |
| budget 16,384 | 15.2% |
| budget 65,536 | 38.6% |

Two readings:

- Against fixed bs=32, length-sorted batching at budget 16,384 cuts waste
  **78.8% → 15.2%**, i.e. roughly **5x less compute burnt on padding**; at
  small budgets the reduction is ~75x.
- The budget curve is not flat: waste **grows with the budget**, because a
  bigger budget spans a wider length range per batch. The knee is the
  interesting operating point — below ~4k the waste is negligible and the
  budget should be chosen for GPU utilisation, not padding.

In [ ]:
# The degenerate case that makes the whole argument: one long document
# lands in a fixed-count batch of short ones.
degenerate = np.array([10] * 31 + [2000])
padded = padded_fixed(degenerate, 32)
real = int(degenerate.sum())
print(f"fixed bs=32:  real={real}  padded={padded}  "
      f"waste_ratio={(padded - real) / padded:.1%}")

# make_batches isolates the long document instead:
texts = ["x" * int(n) for n in degenerate]
batches = list(make_batches(list(enumerate(texts)), 2048))
print(f"make_batches(budget=2048): {[len(b) for b in batches]} items per batch, "
      f"long doc alone={any(len(b) == 1 for b in batches)}")
padded_sorted = padded_budget(degenerate, 2048)
print(f"              real={real}  padded={padded_sorted}  "
      f"waste_ratio={(padded_sorted - real) / padded_sorted:.1%}")

## Degenerate case

31 texts of length 10 plus one of length 2,000 in a single fixed batch of 32:
padded cost **64,000** against **2,310** real tokens — **96.4% waste**. One
stray long document makes every short text in its batch cost 200x its real
length. Length-sorted batching puts the long document in a batch of its own
and the waste collapses to near zero. This single example is the clearest
argument for the approach.